In [22]:
import sys
print(sys.executable)

c:\Users\ASUS\Desktop\DSAI\pythonProject\car-recommendation-system\venv\Scripts\python.exe


In [ ]:
import sys
import pandas as pd
import numpy as np
import os
import shutil
from datetime import datetime

sys.path.append('..')

# Clear cache
pycache_path = '../src/recommender/__pycache__'
if os.path.exists(pycache_path):
    shutil.rmtree(pycache_path)
    print(f"✅ Deleted cache")

# Import
from src.recommender.simplecarrecpmmender import simplecarrecpmmender
print("✅ Imported recommender")

# Load data
print("\n📚 Loading data...")
cars_processed = pd.read_csv('../data/cars_processed.csv')
cars_raw = pd.read_csv('../data/cars_versions_specs.csv')
users_processed = pd.read_csv('../data/users_processed.csv')
users_raw = pd.read_csv('../data/users_dataset.csv')
print(f"   ✅ Loaded {len(cars_processed)} cars and {len(users_raw)} users")

# MLFLOW SETUP
import mlflow

# Create mlflow folder
mlflow_path = os.path.abspath('../mlflow')
os.makedirs(mlflow_path, exist_ok=True)

# Use SQLite
db_path = os.path.join(mlflow_path, 'experiments.db')
mlflow.set_tracking_uri(f'sqlite:///{db_path}')

print(f"\n📊 MLflow database: {db_path}")

# Create experiment
experiment_name = "Simple_Car_Experiments"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"✅ Created new experiment: {experiment_name}")
except:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
    print(f"✅ Using existing experiment: {experiment_name}")

mlflow.set_experiment(experiment_name)

# Create recommender
print("\n🔧 Creating recommender...")
rec = simplecarrecpmmender(cars_processed, cars_raw, users_processed, users_raw)

# Run a simple experiment
print("\n🚀 Running experiment...")
with mlflow.start_run(run_name="first_test"):
    
    # Log parameters
    mlflow.log_param("num_features", len(rec.feature_columns))
    mlflow.log_param("test_date", str(datetime.now()))
    
    # Calculate metrics (test with 3 users)
    scores = []
    for user_id in range(3):
        try:
            recs = rec.recommend(user_id, 3)
            if len(recs) > 0:
                scores.extend(recs['similarity_score'].tolist())
        except:
            pass
    
    avg_score = np.mean(scores) if scores else 0
    mlflow.log_metric("avg_similarity", avg_score)
    
    print(f"   ✅ Average similarity: {avg_score:.2f}%")

print("\n✅ Experiment complete!")
print(f"\n👉 To view results, open a NEW terminal and run:")
print(f"   cd {os.path.dirname(mlflow_path)}")
print(f"   mlflow ui --backend-store-uri sqlite:///mlflow/experiments.db")

✅ Imported recommender

📚 Loading data...
   ✅ Loaded 549 cars and 1000 users

📊 MLflow database: c:\Users\ASUS\Desktop\DSAI\pythonProject\car-recommendation-system\mlflow\experiments.db


2026/03/09 01:54:54 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/09 01:54:54 INFO mlflow.store.db.utils: Updating database tables
INFO:src.recommender.simplecarrecpmmender:📊 Using 20 features for matching
INFO:src.recommender.simplecarrecpmmender:Features: ['body_berline', 'body_suv', 'body_citadine', 'body_compacte', 'body_coupé', 'body_cabriolet', 'body_monospace', 'body_utilitaire', 'body_minibus', 'body_pick up', 'is_electric', 'is_hybrid', 'is_diesel', 'is_essence', 'transmission_boîte_automatique', 'transmission_transmission_intégrale', 'price', 'caractéristiques_nombre_de_places', 'motorisation_puissance_(ch.din)', 'consommation_consommation_mixte']


✅ Created new experiment: Simple_Car_Experiments

🔧 Creating recommender...

🚀 Running experiment...

🔍 User vector shape: (1, 20)
   Sample values: [0. 1. 0. 0. 0.]...
💰 Budget: 40,000 €
   Found 22 cars within budget
📊 Similarity scores: min=0.150, max=0.797

🔍 User vector shape: (1, 20)
   Sample values: [0. 0. 0. 0. 1.]...
💰 Budget: 73,364 €
   Found 154 cars within budget
📊 Similarity scores: min=0.002, max=0.575

🔍 User vector shape: (1, 20)
   Sample values: [0. 1. 0. 0. 0.]...
💰 Budget: 40,000 €
   Found 22 cars within budget
📊 Similarity scores: min=0.012, max=0.749
   ✅ Average similarity: 55.80%

✅ Experiment complete!

👉 To view results, open a NEW terminal and run:
   cd c:\Users\ASUS\Desktop\DSAI\pythonProject\car-recommendation-system
   mlflow ui --backend-store-uri sqlite:///mlflow/experiments.db


In [2]:
# Run multiple experiments
print("\n" + "="*60)
print("RUNNING MULTIPLE EXPERIMENTS")
print("="*60)

# Experiment 1: Basic
with mlflow.start_run(run_name="basic_recommender"):
    mlflow.log_param("type", "basic")
    mlflow.log_param("weights", "none")
    
    scores = []
    for user_id in range(3):
        recs = rec.recommend(user_id, 3)
        scores.extend(recs['similarity_score'].tolist())
    
    mlflow.log_metric("avg_similarity", np.mean(scores))
    print("✅ Logged basic recommender")

# Experiment 2: With weights (just logging different params)
with mlflow.start_run(run_name="weighted_recommender"):
    mlflow.log_param("type", "weighted")
    mlflow.log_param("weight_suv", 2.0)
    mlflow.log_param("weight_hybrid", 3.0)
    
    scores = []
    for user_id in range(3):
        recs = rec.recommend(user_id, 3)
        scores.extend(recs['similarity_score'].tolist())
    
    mlflow.log_metric("avg_similarity", np.mean(scores))
    print("✅ Logged weighted recommender")

print("\n✅ All experiments complete!")


RUNNING MULTIPLE EXPERIMENTS

🔍 User vector shape: (1, 20)
   Sample values: [0. 1. 0. 0. 0.]...
💰 Budget: 40,000 €
   Found 22 cars within budget
📊 Similarity scores: min=0.150, max=0.797

🔍 User vector shape: (1, 20)
   Sample values: [0. 0. 0. 0. 1.]...
💰 Budget: 73,364 €
   Found 154 cars within budget
📊 Similarity scores: min=0.002, max=0.575

🔍 User vector shape: (1, 20)
   Sample values: [0. 1. 0. 0. 0.]...
💰 Budget: 40,000 €
   Found 22 cars within budget
📊 Similarity scores: min=0.012, max=0.749
✅ Logged basic recommender

🔍 User vector shape: (1, 20)
   Sample values: [0. 1. 0. 0. 0.]...
💰 Budget: 40,000 €
   Found 22 cars within budget
📊 Similarity scores: min=0.150, max=0.797

🔍 User vector shape: (1, 20)
   Sample values: [0. 0. 0. 0. 1.]...
💰 Budget: 73,364 €
   Found 154 cars within budget
📊 Similarity scores: min=0.002, max=0.575

🔍 User vector shape: (1, 20)
   Sample values: [0. 1. 0. 0. 0.]...
💰 Budget: 40,000 €
   Found 22 cars within budget
📊 Similarity scores: mi

In [3]:
# reset_mlflow.py
import mlflow

# End any active runs
mlflow.end_run()

print("✅ Active runs ended")

✅ Active runs ended
